In [1]:
!pip install elasticsearch==8.19.1 kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 940.5/940.5 kB 1.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [elasticsearch]0m [elasticsearch]ort]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:

from elasticsearch import Elasticsearch
import os
import pandas as pd



In [7]:
!docker cp search-system-es01-1:/usr/share/elasticsearch/config/certs/ca/ca.crt ./ca.crt

Successfully copied 3.07kB to /workspaces/search-system/search-db/ca.crt


In [8]:
!docker info | grep -i memory
!docker info | grep -i cpu

 Total Memory: 7.758GiB
 CPUs: 2


In [9]:

!free -h

               total        used        free      shared  buff/cache   available
Mem:           7.8Gi       5.9Gi       155Mi        63Mi       2.3Gi       1.9Gi
Swap:             0B          0B          0B


In [14]:
!df -h

Filesystem      Size  Used Avail Use% Mounted on
overlay          32G   21G  9.0G  70% /
tmpfs            64M     0   64M   0% /dev
shm              64M  4.0K   64M   1% /dev/shm
/dev/root        29G   22G  7.7G  74% /vscode
/dev/loop4       32G   21G  9.0G  70% /workspaces
/dev/sdb1        44G  4.0G   38G  10% /tmp


In [13]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("phamtheds/news-dataset-vietnameses")
print("Path to dataset files:", path)

100%|██████████| 308M/308M [00:17<00:00, 18.9MB/s] 

Extracting files...


Path to dataset files: /home/codespace/.cache/kagglehub/datasets/phamtheds/news-dataset-vietnameses/versions/1


In [11]:
client = Elasticsearch(
    hosts=["https://localhost:9200"],  # Địa chỉ Elasticsearch
    basic_auth=("elastic", "elastic"),
    request_timeout=60,
    ca_certs="./ca.crt"
)
client.info()

ObjectApiResponse({'name': 'es01', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'cbluafqwRbiqoWRkIigyjw', 'version': {'number': '8.19.4', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'aa0a7826e719b392e7782716b323c4fb8fa3b392', 'build_date': '2025-09-16T22:06:03.940754111Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [15]:

def get_all_file_names(folder_path):
    try:
        # List all files in the folder and remove ".json" extension
        file_names = [
            os.path.splitext(file)[0] for file in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, file)) and (file.endswith(".ndjson") or file.endswith(".csv"))
        ]
        return file_names
    except FileNotFoundError:
        print(f"The folder '{folder_path}' does not exist.")
        return []

folder_path = path
# folder_path = "data"
file_names = get_all_file_names(folder_path)
print("Files in folder:", file_names)

Files in folder: ['Dataset_articles_NoID']


In [16]:
# Tạo index template
import json
with open('index_template.json', 'r') as f:
    template = json.load(f)
client.indices.put_index_template(
    name="baolaodong_template",
    index_patterns=template['index_patterns'],
    template=template['template']
)

ObjectApiResponse({'acknowledged': True})

In [17]:
import ast
import re
def safe_literal_eval(val):
    """Chuyển đổi string list thành Python list an toàn"""
    if pd.isna(val) or val == '':
        return []
    try:
        # Xử lý trường hợp có dấu ngoặc vuông
        if val.startswith('[') and val.endswith(']'):
            return ast.literal_eval(val)
        else:
            # Nếu không phải list, coi như single value
            return [val.strip()]
    except (ValueError, SyntaxError):
        # Nếu có lỗi, split bằng dấu phẩy
        return [item.strip().strip("'\"") for item in val.split(',')]

In [ ]:
from elasticsearch import helpers
import traceback
# Process NDJSON files

# Create index (matches wikipedia-people* pattern)

INDEX_NAME = "articles-csv"
if not client.indices.exists(index=INDEX_NAME):
    client.indices.create(index=INDEX_NAME)
    print(f"Index '{INDEX_NAME}' created successfully")

batch_size = 1
actions = []

# for file_name in os.listdir(ndjson_dir):
#     if file_name.endswith(".ndjson"):
#         file_path = os.path.join(ndjson_dir, file_name)
path = 'data'
with open(f"{path}/{file_names[5]}.ndjson", "r", encoding="utf-8") as f:
    for line in f:

        try:
            doc = json.loads(line.strip())
        except json.JSONDecodeError:
            # print(f"Skipping malformed JSON in {file_name}")
            continue
        # print(doc)
        # break

        # Build full_text from sections
        full_text = extract_text_from_sections(doc.get("article_sections", []))
        # print(full_text)
        # break

        # Sử dụng giá trị mặc định nếu thiếu
        if not full_text or not full_text.strip():
            full_text = "No content available"

        # Prepare ES document
        es_doc = {
            "_index": INDEX_NAME,
            "_id": str(doc.get("identifier")),  # Use identifier as doc ID
            "_source": {
                "identifier": doc.get("identifier"),
                "name": doc.get("name"),
                "description": doc.get("description"),
                "abstract": doc.get("abstract"),
                "full_text": full_text,
                "image": doc.get("image"),
                "categories": [cat["name"] for cat in doc.get("categories", [])],
                "infobox": doc.get("infobox", []),
                "date_modified": doc.get("date_modified"),
                "url": doc.get("url"),
                "name_embedding": doc.get("name_embedding"),
                "abstract_embedding": doc.get("abstract_embedding"),
                "full_text_embedding": doc.get("full_text_embedding"),
                "main_entity": doc.get("main_entity"),
            }
        }
        actions.append(es_doc)

        # Bulk index when batch is full
        if len(actions) >= batch_size:
            try:
                helpers.bulk(client, actions)
                # print(f"Indexed {len(actions)} documents from {file_names[5]}")
                actions = []

            except Exception as e:
                print(f"Error indexing batch: {e}")
                print(actions)

# Index any remaining documents
if actions:
    try:
        helpers.bulk(client, actions)
        print(f"Indexed final {len(actions)} documents")
    except Exception as e:
        print(f"Error indexing final batch: {e}")
        print(actions)

print("Ingestion complete!")

Index 'wikipedia-people-sample-embedding' created successfully
Ingestion complete!


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")